# Tier 1 — Smoke Test Detector (YOLOv10s / YOLO26n / RT-DETR)

Tujuan: **membuktikan pipeline deteksi orang jalan di GPU Colab** sebelum lanjut ke skenario
S1 (evaluasi detector pada CrowdHuman), S2 (tracker di MOT20/DanceTrack), S3 (counting logic),
S4 (end-to-end).

**Batasan jujur notebook ini:**
- Inference-only. Tidak ada training, tidak ada fine-tuning, tidak ada eksperimen kuantitatif.
- Detector default = `yolov10s` (S003 — NeurIPS 2024, peer-reviewed).
- `yolo26n` (S001/S002) disediakan sebagai opsi perbandingan — bukan bukti SOTA akademik.
- Sample video diambil dari URL publik pendek. Tidak ada klaim dataset coverage.

**Cara pakai di Colab:**
1. Buka https://colab.research.google.com → upload notebook ini (File → Upload notebook).
2. Runtime → Change runtime type → **T4 GPU**.
3. Jalankan semua cell berurutan. Cell terakhir akan menyimpan hasil ke `/content/drive/MyDrive/hibah-riset/` (jika di-mount).
4. Setelah selesai, salin folder `outputs/` ke repo lokal, commit, push.

## 0. Mount Drive (opsional, untuk simpan hasil permanen)

In [ ]:
# Uncomment kalau mau simpan hasil ke Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
#
# import os
# OUTPUT_ROOT = '/content/drive/MyDrive/hibah-riset/experiments/s1_detector_smoke'
# os.makedirs(OUTPUT_ROOT, exist_ok=True)

OUTPUT_ROOT = '/content/hibah-riset-smoke'  # fallback ephemeral di Colab VM
import os
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('Output root:', OUTPUT_ROOT)

## 1. Install deps

In [ ]:
%pip install -q ultralytics>=8.3.0 opencv-python>=4.8.0 numpy>=1.24.0 tqdm>=4.65.0

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 2. Clone repo agar dapat mengakses `src/` dan `configs/`

Pakai HTTPS publik. Tidak butuh token karena repo `feb027/hibah-riset` public.

In [ ]:
import os, sys, pathlib

REPO_URL = 'https://github.com/feb027/hibah-riset.git'
REPO_DIR = '/content/hibah-riset'

if not pathlib.Path(REPO_DIR).exists():
    !git clone --depth=1 $REPO_URL $REPO_DIR
else:
    print('Repo already present at', REPO_DIR)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.'))[:20])

## 3. Siapkan sample video

Kita pakai video pendek dari Pexels (CC0) atau sample bawaan ultralytics.
Kalau gagal download, kita generate synthetic video (frames berisi persegi).

In [ ]:
import urllib.request, pathlib

DATA_DIR = pathlib.Path('/content/sample_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
video_path = DATA_DIR / 'crowd_sample.mp4'

# Opsi A: download video pendek Pexels (CC0). Ganti URL jika mati.
SAMPLE_URLS = [
    'https://videos.pexels.com/video-files/3192301/3192301-uhd_3840_2160_25fps.mp4',  # crowd walking
    'https://videos.pexels.com/video-files/3209661/3209661-uhd_2560_1440_25fps.mp4',  # pedestrian
]

if not video_path.exists():
    for url in SAMPLE_URLS:
        try:
            print('Trying:', url)
            urllib.request.urlretrieve(url, video_path)
            if video_path.stat().st_size > 100_000:
                print('Downloaded', video_path.stat().st_size, 'bytes')
                break
        except Exception as e:
            print('Failed:', e)
    else:
        print('All sample URLs failed; will generate synthetic video instead.')

if not video_path.exists() or video_path.stat().st_size < 100_000:
    import cv2, numpy as np
    # Synthetic 10-second video with moving rectangles (simulasi orang).
    fps, w, h, n = 25, 640, 480, 250
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(video_path), fourcc, fps, (w, h))
    rng = np.random.default_rng(42)
    n_people = 6
    pos = rng.uniform([0, 0], [w - 60, h - 120], size=(n_people, 2))
    vel = rng.uniform([-2, -2], [2, 2], size=(n_people, 2))
    for t in range(n):
        frame = np.full((h, w, 3), 200, dtype=np.uint8)
        for i in range(n_people):
            pos[i] += vel[i]
            pos[i] = np.clip(pos[i], 0, [w - 60, h - 120])
            vel[i] = np.where((pos[i] <= 0) | (pos[i] >= [w - 60, h - 120]), -vel[i], vel[i])
            x, y = pos[i]
            cv2.rectangle(frame, (int(x), int(y)), (int(x + 60), int(y + 120)), (50, 50, 200), -1)
            cv2.putText(frame, f'p{i}', (int(x) + 10, int(y) + 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        writer.write(frame)
    writer.release()
    print('Synthetic video written:', video_path, video_path.stat().st_size, 'bytes')

print('video_path:', video_path)

## 4. Jalankan smoke test (default: YOLOv10s)

In [ ]:
from src.pipeline import run_smoke_test
import json

summary = run_smoke_test(
    video_path=str(video_path),
    output_dir=OUTPUT_ROOT,
    detector_name='yolov10s',
    confidence_threshold=0.25,
    iou_threshold=0.45,
    max_frames=200,   # batasi supaya cepat di Colab free
    device='cuda:0' if torch.cuda.is_available() else 'cpu',
)
print(json.dumps(summary.__dict__, indent=2))

## 5. Opsional: bandingkan dengan detector lain

Catatan: hanya deskriptif, BUKAN benchmark akademis. Sumber sitasi sudah jelas di `src/detector.py`.

In [ ]:
from src.pipeline import run_smoke_test

for det in ['yolov11n', 'yolo26n']:
    print('\n===', det, '===')
    try:
        s = run_smoke_test(
            video_path=str(video_path),
            output_dir=OUTPUT_ROOT,
            detector_name=det,
            confidence_threshold=0.25,
            iou_threshold=0.45,
            max_frames=200,
            device='cuda:0' if torch.cuda.is_available() else 'cpu',
        )
        print('source_id:', s.source_id, '| fps_avg:', round(s.fps_avg, 2),
              '| mean_persons:', round(s.mean_persons_per_frame, 2))
    except Exception as e:
        print('Skipped', det, ':', e)

## 6. Plot count per frame (diagnostik, bukan bukti SOTA)

In [ ]:
import csv, matplotlib.pyplot as plt
import os

csv_path = os.path.join(OUTPUT_ROOT, 'per_frame_yolov10s.csv')
if os.path.exists(csv_path):
    xs, ys, lats = [], [], []
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            xs.append(int(row['frame_index']))
            ys.append(int(row['person_count']))
            lats.append(float(row['latency_ms']))
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(xs, ys, label='yolov10s person count')
    axes[0].set_ylabel('Persons / frame')
    axes[0].grid(alpha=0.3)
    axes[0].legend()
    axes[1].plot(xs, lats, color='orange', label='latency_ms')
    axes[1].set_ylabel('Latency (ms)')
    axes[1].set_xlabel('Frame index')
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    out_png = os.path.join(OUTPUT_ROOT, 'diagnostic_yolov10s.png')
    plt.savefig(out_png, dpi=120)
    print('Saved diagnostic plot to', out_png)
else:
    print('CSV not found at', csv_path)

## 7. Ringkasan & langkah lanjut

Setelah smoke test lulus, **langkah aman berikutnya** (Tier 2):

1. Tambah `scripts/download_mot17_mini.py` untuk unduh subset MOT17 (~150 MB).
2. Integrasikan OC-SORT via `from ultralytics import YOLO` dengan argumen `tracker='botsort.yaml'` atau pakai `boxmot`.
3. Hitung IDF1/HOTA pada subset mini.

**Tidak dijalankan di Tier 1:** training, fine-tuning, klaim angka SOTA, DiffMOT.

**Yang harus dilaporkan ke dosen**: jumlah frame yang diproses, FPS rata-rata,
FPS p95, mean person count, dan path ke video annotated + CSV + JSON.
Jangan klaim akurasi atau superiority.